# Exercicis — Agrupar sense etiquetes amb k-means

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Tots els exercicis que has fet fins ara tenien la resposta a la columna `y`, i el model aprenia
a imitar-la. Aquí **la `y` no la toques**: l'algorisme només veu `X`. Les etiquetes reals, quan
n'hi ha, les fas servir **després**, i només per mirar fins a quin punt els grups que ha trobat
k-means s'hi assemblen.

La teoria és a `01_teoria/ML_06_kmeans.ipynb`. Els dos primers exercicis són de càlcul a mà
sobre una taula de set punts, i porten el número que t'ha de sortir. La resta són sobre dades de
veritat, i allà **no hi ha número**: hi ha comprovacions que et diuen si el procediment és
correcte, i la conclusió és feina teva.

Fixa `random_state=42` i `n_init=10` a tots els `KMeans` on l'enunciat no digui una altra cosa.

## Les eines que necessites

Aquí les tens llistades, no encadenades: quina va abans de quina i què li passes a cada una és la
feina dels exercicis.

- De NumPy: el *broadcasting* amb `X[:, None, :]`, `.sum(axis=...)`, `.argmin(axis=1)`,
  `.mean(axis=0)`, les màscares booleanes (`X[etiquetes == j]`), `np.bincount`, `np.unique`,
  `np.allclose`, `np.array_equal`, `np.random.default_rng`.
- De `sklearn.cluster`: `KMeans`.
- De `sklearn.preprocessing`: `StandardScaler`.
- De `sklearn.metrics`: `silhouette_score` i `adjusted_rand_score`.
- De `sklearn.decomposition`: `PCA`.
- De `sklearn.datasets`: `load_wine`, `load_digits` i `load_sample_image`.
- De pandas: `pd.crosstab`.
- De matplotlib: `plt.plot`, `plt.subplots`, `.imshow`.

Totes, menys `load_sample_image`, surten fent-se servir a `ML_06_kmeans.ipynb`: el
*broadcasting* de les distàncies a la secció 2, `KMeans` i els seus atributs a la 6, la silueta
a la 8, `pd.crosstab` i l'ARI a la 9. `StandardScaler` i `PCA` ja els coneixes; si et cal
repassar-los, són a `EX_00_caixa_eines.ipynb` i a `MA_06_pca_vectors_propis.ipynb`.

## Els objectes que manegues, i què els pots demanar

**L'`ndarray` de NumPy** — `X`, `M`, les etiquetes, `cluster_centers_`, la foto:
- `.shape` — la forma. Un conjunt de dades és `(n, d)`; uns centres, `(k, d)`; les distàncies
  de cada punt a cada centre, `(n, k)`
- `X[:, None, :]` — afegeix un eix al mig: de `(n, d)` passa a `(n, 1, d)`. Restant-li un array
  de forma `(1, k, d)`, el *broadcasting* et dona `(n, k, d)`
- `.sum(axis=2)`, `.mean(axis=0)`, `.min(axis=1)` — l'eix és el que **desapareix**
- `.argmin(axis=1)` — la **posició** del mínim de cada fila, no el valor. Amb una matriu de
  distàncies `(n, k)`, és el grup de cada punt
- `M[etiquetes]` — indexar amb un array: una fila de `M` per cada punt, la del seu grup
- `X[etiquetes == j]` — només les files del grup `j`
- `.reshape(...)` — la mateixa informació amb una altra forma: `(64,)` a `(8, 8)`, o una foto
  `(alt, ample, 3)` a una taula `(alt*ample, 3)` i tornar
- `np.bincount(etiquetes)` — quants punts hi ha a cada grup
- `np.unique(a, axis=0)` — les files diferents d'un array

**El `KMeans`** — es crea amb els paràmetres i s'entrena amb `.fit(X)`. **No rep `y`**:
- paràmetres: `n_clusters`, `init` (`"k-means++"` o un array amb els centres inicials),
  `n_init`, `algorithm`, `tol`, `random_state`
- `.fit(X)` — entrena; torna el mateix objecte
- `.fit_predict(X)` — entrena i torna les etiquetes, tot d'un cop
- `.predict(X_nou)` — assigna cada fila al centre més proper, **sense moure cap centre**
- `.transform(X)` — la matriu `(n, k)` de distàncies de cada punt a cada centre, **sense elevar
  al quadrat**
- `.cluster_centers_` — els centres, `(k, d)`
- `.labels_` — el grup de cada punt de l'entrenament. **El número del grup no vol dir res**
- `.inertia_` — la $J$ final: suma de distàncies **al quadrat** de cada punt al seu centre
- `.n_iter_` — quantes rondes ha fet

**L'`StandardScaler`** — `.fit_transform(X)` torna les dades amb mitjana 0 i desviació 1 a cada
columna; `.mean_` i `.scale_` són el que ha après.

**`silhouette_score(X, etiquetes)` i `adjusted_rand_score(y_real, etiquetes)`** — no són
objectes, són **funcions**: reben arrays i tornen un sol número (`float`). La silueta no
necessita cap etiqueta real; l'ARI sí, i no li importa com estiguin numerats els grups.

**El `DataFrame` que torna `pd.crosstab(a, b)`** — una fila per cada valor de `a`, una columna
per cada valor de `b`, i a cada casella quantes vegades surt la parella:
- `.to_numpy()` — la taula com a `ndarray`
- `.sum().sum()` — el total de la taula: ha de ser el nombre de files que has comptat
- `rownames=[...]`, `colnames=[...]` — per posar nom als eixos quan la crides

**El `Bunch` de `load_wine()` i `load_digits()`** — `.data`, `.target`, `.feature_names`,
`.target_names`, i `.images` (només `digits`, les mateixes dades en forma `(1797, 8, 8)`).

**`load_sample_image("china.jpg")`** — no torna un `Bunch`, torna directament un `ndarray` de
forma `(alt, ample, 3)` amb enters de 0 a 255: cada píxel són tres números, vermell, verd i
blau. Ve dins de scikit-learn i no cal baixar res.

**El `PCA`** — `.fit_transform(X)` projecta; `.transform(...)` projecta **amb la mateixa
transformació** qualsevol altre array de les mateixes columnes, per exemple uns centres.

**L'`Axes` de matplotlib** — `.imshow(imatge, cmap="gray")`, `.set_title(text)`, `.axis("off")`;
i `axs.ravel()` aplana la graella d'eixos de `plt.subplots(files, columnes)`.

Les dues regles de sempre: **un atribut va sense parèntesis i un mètode amb parèntesis**, i
**el guió baix final (`labels_`, `inertia_`) vol dir «això ho he après de les dades»**: no
existeix abans del `.fit()`. Per interrogar qualsevol objecte, `type()`, `dir()` i el TAB de
`FO_00_objectes_i_autocompletar.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# La taula de joguina dels exercicis 1 i 2: set punts i dos centres de partida
X_joguina = np.array([[1.0, 1.0],
                      [1.5, 2.0],
                      [3.1, 3.9],
                      [5.0, 7.0],
                      [3.5, 5.0],
                      [4.5, 5.0],
                      [3.5, 4.5]])
M_joguina = np.array([[1.0, 1.0],
                      [5.0, 7.0]])

## Exercici 1 — Distàncies, assignació i inèrcia, a mà

Amb `X_joguina` i `M_joguina`, i **sense cap bucle**:

1. Calcula la matriu `D` de distàncies **al quadrat** de cada punt a cada centre, amb el
   *broadcasting* de la secció 2 de la teoria.
2. Assigna cada punt al centre més proper (`etiquetes`).
3. Calcula la inèrcia $J = \sum_i \|x_i - \mu_{c(i)}\|^2$ amb `M_joguina[etiquetes]`.

**Com saps que ho has fet bé**: `D` ha de tenir forma `(7, 2)`, amb `D[2]` igual a
`[12.82, 13.22]`. Les etiquetes han de ser `[0 0 0 1 1 1 1]` i la inèrcia, **33.07**.

In [ ]:
# Exercici 1

# D: resta X_joguina[:, None, :] menys M_joguina[None, :, :], eleva al quadrat
# i suma sobre l'eix de les columnes


# etiquetes: la posicio del minim de cada fila de D


# J: resta a cada punt el centre del seu grup (M_joguina[etiquetes]),
# eleva al quadrat i suma-ho tot

## Exercici 2 — L'algorisme de Lloyd, sobre la joguina

**a)** Fes **una** actualització: cada centre nou és la mitjana dels punts del seu grup (fes-ho
amb una màscara `X_joguina[etiquetes == j]` per a cada `j`). Calcula la inèrcia amb **les
mateixes etiquetes** i els centres nous.

**b)** Escriu un bucle que alterni assignar i actualitzar fins que les etiquetes deixin de
canviar, guardant la inèrcia després de cada mig pas en una llista `historial`.

**c)** Entrena `KMeans(n_clusters=2, init=M_joguina, n_init=1, algorithm="lloyd", tol=0)` amb
`X_joguina` i compara els seus `cluster_centers_` i `inertia_` amb els teus.

**Com saps que ho has fet bé**: a l'apartat a), els centres nous han de ser
`[[1.8667, 2.3], [4.125, 5.375]]` i la inèrcia, **12.1217**, que no pot ser més gran que la de
l'exercici 1 (per què? és la secció 4 de la teoria). Al final del bucle, els centres han de ser
`[[1.25, 1.5], [3.92, 5.08]]` amb inèrcia **8.581**; `historial` no pot pujar mai
(`np.all(np.diff(historial) <= 1e-9)` ha de donar `True`), i amb `KMeans` la diferència de
centres ha de ser de l'ordre de $10^{-16}$ i la inèrcia la mateixa.

In [ ]:
# Exercici 2
from sklearn.cluster import KMeans

# a) centres nous: per a cada j in range(2), la mitjana (axis=0) de X_joguina[etiquetes == j]
#    i la inercia amb les MATEIXES etiquetes de l'exercici 1


# b) bucle: assigna, guarda J, mira si les etiquetes han canviat (np.array_equal),
#    actualitza, guarda J... fins que no canviin


# c) KMeans(n_clusters=2, init=M_joguina, n_init=1, algorithm="lloyd", tol=0).fit(X_joguina)
#    compara .cluster_centers_ i .inertia_ amb els teus

## Exercici 3 — El vi, sense etiquetes: quants grups?

Carrega `load_wine()`. Fes servir **només** `wine.data` (178 vins, 13 columnes químiques): fes
com si `wine.target` no existís. Escala-ho amb `StandardScaler`.

Per a cada $k$ de 1 a 10, entrena un `KMeans(n_clusters=k, n_init=10, random_state=42)` i
guarda'n la `inertia_`. Per a cada $k$ de 2 a 10, calcula'n també la `silhouette_score`.
Dibuixa les dues corbes, una al costat de l'altra.

Després, en una cel·la de text, **tria una $k$ i justifica-la** amb els dos gràfics.

**Com saps que ho has fet bé**: comprova que les dades escalades tenen mitjana ≈ 0 i desviació
≈ 1 a les 13 columnes (`np.allclose(..., 0)` i `np.allclose(..., 1)`); que la inèrcia **no
puja mai** quan $k$ creix, com demostra la secció 8 de la teoria; i que totes les siluetes són
entre −1 i 1. Si la inèrcia et puja en algun pas, revisa el `n_init`. La $k$ que triïs és la teva
conclusió: no hi ha cap número aquí que te la doni.

In [ ]:
# Exercici 3
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

wine = load_wine()

# escala wine.data amb StandardScaler().fit_transform(...) i desa-ho a X_vi
# comprova mitjana ~0 i desviacio ~1 per columna (axis=0)


# per a k de 1 a 10: KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_vi)
# guarda .inertia_ a una llista; si k >= 2, guarda silhouette_score(X_vi, model.labels_)


# comprova que la inercia no puja: np.all(np.diff(inercies) <= 1e-9)
# i que les siluetes son entre -1 i 1


# dues grafiques amb plt.subplots(1, 2): inercia contra k, silueta contra k

*La teva tria de $k$, i per què:*

## Exercici 4 — El model per dins

Entrena un `KMeans` amb la $k$ que has triat a l'exercici 3 sobre `X_vi`, i comprova, amb
**una línia cadascuna**, que els seus atributs compleixen el que diu la teoria:

1. Cada vi està assignat al centre **més proper**: `labels_` és igual a l'`argmin` de cada fila
   de `.transform(X_vi)`.
2. Cada centre és **la mitjana** dels vins del seu grup.
3. `inertia_` és la suma de les distàncies **al quadrat** de cada vi al seu centre.
4. Cada vi és en un grup i només en un: els recomptes de `np.bincount(labels_)` sumen 178.

**Com saps que ho has fet bé**: les quatre comprovacions donen `True` (fes servir
`np.array_equal` per a les etiquetes i `np.allclose` per als números amb decimals). Si la 2 et
dona `False`, mira si has fet la mitjana per l'eix correcte.

In [ ]:
# Exercici 4
k_triada = None   # la k que has triat a l'exercici 3

# km_vi = KMeans(n_clusters=k_triada, n_init=10, random_state=42).fit(X_vi)

# 1. etiquetes == argmin de les distancies


# 2. per a cada j: centre j == mitjana de X_vi[km_vi.labels_ == j]


# 3. inertia_ == suma del minim de cada fila de transform, al quadrat


# 4. np.bincount(km_vi.labels_).sum() == len(X_vi)

## Exercici 5 — Ara sí, les etiquetes: s'assemblen els grups als cultivars?

`wine.target` diu de quin dels tres cultivars (varietats de raïm) és cada vi. k-means no l'ha
vist. Compara-hi els grups del teu `km_vi`:

1. Fes la taula de grups contra cultivars amb `pd.crosstab`.
2. Calcula l'`adjusted_rand_score` entre `wine.target` i les etiquetes.
3. **Ara sense escalar**: entrena el mateix `KMeans` sobre `wine.data` tal com ve i calcula'n
   també l'ARI.

En una cel·la de text, explica què et diu la taula, què et diu la comparació entre els dos ARI,
i **per què** (mira la desviació típica de cada columna de `wine.data`, i la secció 10 de la
teoria).

**Com saps que ho has fet bé**: la taula ha de sumar 178; l'ARI ha de donar **exactament el
mateix** si renombres els grups amb una permutació (per exemple `np.array([2, 0, 1])[etiquetes]`,
si tens 3 grups), perquè el número d'un grup no vol dir res; i l'ARI de `wine.target` contra ell
mateix ha de ser 1. Els dos ARI que compares són la teva troballa.

In [ ]:
# Exercici 5
from sklearn.metrics import adjusted_rand_score

# 1. pd.crosstab(km_vi.labels_, wine.target, rownames=["grup"], colnames=["cultivar"])
#    i comprova que suma 178 (.sum().sum())


# 2. ARI entre wine.target i km_vi.labels_
#    i comprova que no canvia amb els grups renombrats


# 3. el mateix KMeans sobre wine.data sense escalar, i el seu ARI
#    mira tambe wine.data.std(axis=0)

*Què et diu la taula, què et diuen els dos ARI, i per què:*

## Exercici 6 — Dígits sense saber quina xifra és cada imatge

Carrega `load_digits()` i agrupa `digits.data` (1797 imatges de 64 píxels) amb
`KMeans(n_clusters=10, n_init=10, random_state=42)`. **No escalis**: tots els píxels van a la
mateixa escala, de 0 a 16, i la diferència entre píxels és informació (és el mateix argument
que al PCA de `MA_06`).

1. Dibuixa els 10 centres com a imatges de 8×8, en una graella de 2×5. Cada centre és un punt de
   64 columnes, o sigui una imatge.
2. Fes la taula de grups contra `digits.target` amb `pd.crosstab` i calcula l'ARI.
3. Projecta les dades a 2 dimensions amb `PCA(n_components=2)`, pinta cada punt del color del seu
   grup, i projecta-hi també els 10 centres **amb el mateix PCA** (`.transform`).

En una cel·la de text, digues quines xifres troba bé k-means i quines barreja, i per què creus
que és així mirant els centres dibuixats.

**Com saps que ho has fet bé**: `cluster_centers_` ha de tenir forma `(10, 64)`; tots els seus
valors han de ser entre 0 i 16 (un centre és una mitjana de píxels de 0 a 16, i una mitjana no
pot sortir del rang dels seus valors). Compara amb una tolerància, `C.min() > -1e-9` i
`C.max() < 16 + 1e-9`: hi pot haver un $-10^{-15}$ que no és cap error, és l'arrodoniment dels
decimals; i la taula ha de sumar 1797. Quines xifres es barregen és
la teva troballa.

In [ ]:
# Exercici 6
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()

# km_dig = KMeans(n_clusters=10, n_init=10, random_state=42).fit(digits.data)
# comprova la forma de cluster_centers_ i que els valors son entre 0 i 16
# (amb tolerancia: C.min() > -1e-9 i C.max() < 16 + 1e-9)


# 1. graella 2x5 amb plt.subplots: a cada eix, centre.reshape(8, 8) amb imshow(cmap="gray")
#    i de titol el numero del grup


# 2. pd.crosstab(km_dig.labels_, digits.target, ...) i adjusted_rand_score


# 3. PCA(n_components=2).fit(digits.data): transform de les dades i dels centres
#    scatter de les dades amb c=km_dig.labels_, cmap="tab10"; els centres amb marker="X"

*Quines xifres troba bé k-means, quines barreja, i per què:*

## Exercici 7 (opcional) — Una foto amb pocs colors

Una foto en color també és una taula: cada píxel és una fila de 3 columnes (vermell, verd,
blau). Si agrupes els píxels amb k-means en $k$ grups i substitueixes cada píxel pel **centre
del seu grup**, tens la mateixa foto pintada amb només $k$ colors.

1. Carrega `load_sample_image("china.jpg")`, divideix per 255 (perquè els colors vagin de 0 a 1)
   i converteix-la en una taula de `(alt*ample, 3)` amb `.reshape(-1, 3)`.
2. Són més de 270.000 píxels: entrena el `KMeans` amb **5000 píxels a l'atzar**
   (`np.random.default_rng(42).choice(...)`, sense repetir) i després fes `.predict` de **tots**.
3. Reconstrueix la foto amb `cluster_centers_[etiquetes]` i torna-la a la forma original amb
   `.reshape(foto.shape)`.
4. Fes-ho per a $k$ = 2, 4, 8, 16 i 32 (amb `n_init=4` n'hi ha prou), dibuixa les cinc fotos
   al costat de l'original, i calcula per a cada $k$ l'error RMSE entre l'original i la
   reconstruïda: $\sqrt{\text{mitjana de les diferències al quadrat}}$.

Respon: a partir de quina $k$ deixes de veure la diferència? Coincideix amb el que diu l'RMSE?

**Com saps que ho has fet bé**: cada foto reconstruïda ha de tenir **com a molt $k$ colors
diferents** (`len(np.unique(taula_reconstruida, axis=0)) <= k`), la mateixa forma que
l'original, i l'RMSE no hauria de pujar quan $k$ creix.

In [ ]:
# Exercici 7 (opcional)
from sklearn.datasets import load_sample_image

# foto = load_sample_image("china.jpg") / 255
# pixels = foto.reshape(-1, 3)
# mostra = pixels[np.random.default_rng(42).choice(len(pixels), size=5000, replace=False)]

# per a k in [2, 4, 8, 16, 32]:
#     KMeans(n_clusters=k, n_init=4, random_state=42).fit(mostra)
#     etiquetes de TOTS els pixels amb .predict(pixels)
#     reconstruida = cluster_centers_[etiquetes]
#     comprova colors diferents <= k, i calcula l'RMSE
#     desa reconstruida.reshape(foto.shape) per dibuixar-la